Prueba de modelos clásicos Regresión Logística y SVMs aplicando PCA a los datasets normalizados. Resultados negativos, no merece la pena hacer más
pruebas por aquí, voy a pasar directamente a selección de características por filtrado, wrapper y ensembles.

In [7]:
import pandas as pd
import numpy as np

SEED = 777
np.random.seed(SEED)


In [ ]:
from sklearn.model_selection import train_test_split

# cargo el dataset básico primero para separarlo en train y test
basic = pd.read_csv('../data/scaled/data_basic.csv')


# separación de características y objetivo
X_basic = basic.drop('Variable de Salida', axis=1)
y_basic = basic['Variable de Salida']

# separación en train y test
X_basic_train, X_basic_test, y_basic_train, y_basic_test = train_test_split(X_basic, y_basic, test_size=0.2, random_state = SEED, shuffle=True, stratify=y_basic)

# uno de nuevo y los meto ya en el 

basic = (pd.concat([X_basic_train, y_basic_train], axis=1), pd.concat([X_basic_test, y_basic_test], axis=1))


In [ ]:
# cargar el resto de los datos en un diccionario de datasets, donde la clave es el tipo de trato y el valor una tupla (train, test)

datasets = {
    'basic': basic,
    'max_abs': (pd.read_csv('../data/scaled/data_trainmax_abs.csv'), pd.read_csv('../data/scaled/data_testmax_abs.csv')),
    'min_max': (pd.read_csv('../data/scaled/data_trainmin_max.csv'), pd.read_csv('../data/scaled/data_testmin_max.csv')),
    'robust': (pd.read_csv('../data/scaled/data_trainrobust.csv'), pd.read_csv('../data/scaled/data_testrobust.csv')),
    'standard': (pd.read_csv('../data/scaled/data_trainstandard.csv'), pd.read_csv('../data/scaled/data_teststandard.csv'))
}

In [10]:
from sklearn.decomposition import PCA

# santi ha estado probando con todas las variables y, sorpresa, parece que sobreajusta, así que voy a probar a aplicar PCA a todos los datasets escalados

datasets_pca = {}
modelos_pca = {}    # muy importante para transformar datos nuevos si fuera neceario !!!!!!

for nombre, (train, test) in datasets.items():
    X_train = train.drop('Variable de Salida', axis=1)
    y_train = train['Variable de Salida']

    X_test = test.drop('Variable de Salida', axis=1)
    y_test = test['Variable de Salida']

    pca = PCA(n_components=0.95, random_state=42)

    # fit transform para train y solo transform para test como siempre
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    # esta vez guardamos los 4 elementos
    datasets_pca[nombre] = (X_train_pca, X_test_pca, y_train, y_test)

    # guardo el modelo pca con el fit hecho
    modelos_pca[nombre] = pca
    


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# como método lineal voy a probar regresión logistica e igual alguna variante

# creo un diccionario con los modelos logísticos
modelos_rl = {}

for nombre, (X_train, X_test, y_train, y_test,) in datasets_pca.items():
    print(nombre)
    rl = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')

    rl.fit(X_train, y_train)

    modelos_rl[nombre] = rl

    predicciones = rl.predict(X_test)

    print(confusion_matrix(y_test, predicciones))
    print(classification_report(y_test, predicciones))





basic
[[100  63]
 [ 19  24]]
              precision    recall  f1-score   support

           0       0.84      0.61      0.71       163
           1       0.28      0.56      0.37        43

    accuracy                           0.60       206
   macro avg       0.56      0.59      0.54       206
weighted avg       0.72      0.60      0.64       206

max_abs
[[97 66]
 [15 28]]
              precision    recall  f1-score   support

           0       0.87      0.60      0.71       163
           1       0.30      0.65      0.41        43

    accuracy                           0.61       206
   macro avg       0.58      0.62      0.56       206
weighted avg       0.75      0.61      0.64       206

min_max
[[95 68]
 [16 27]]
              precision    recall  f1-score   support

           0       0.86      0.58      0.69       163
           1       0.28      0.63      0.39        43

    accuracy                           0.59       206
   macro avg       0.57      0.61      0.54  

### Resultados de Regresión Logística para PCA:

Los resultados esperados, sin el parámetro de class_weight el accuracy se dispara pero porque clasifica todos los datos como NOK. Al poner el parámetro el accuracy cae al 62% pero mejora el f1-score bastante. En ambos casos los resultados son muy mediocres, claramente el problema no es lineal

In [17]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# ahora vamos a probar varios modelos no lineales empezando por SVM

# parámetros a probar
param_grid = [
    {
        'kernel': ['rbf'],
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 0.1, 0.01, 0.001]
    },
    {
        'kernel': ['poly'],
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 0.1],
        'degree': [2, 3]
    }
]

modelos_svc = {}

for nombre, (X_train, X_test, y_train, y_test) in datasets_pca.items():
    svc = SVC(class_weight='balanced', max_iter=50000)
    
    clf = GridSearchCV(svc, param_grid, n_jobs=-1, cv=5, verbose=0, refit=True, scoring='f1_macro')
    clf.fit(X_train, y_train)

    # al poner refit = True, clf actúa directamente como mejor modelo
    modelos_svc[nombre] = clf

    predicciones = clf.predict(X_test)

    print(nombre)
    print(confusion_matrix(y_test, predicciones))
    print(classification_report(y_test, predicciones))





/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/py

basic
[[120  43]
 [ 27  16]]
              precision    recall  f1-score   support

           0       0.82      0.74      0.77       163
           1       0.27      0.37      0.31        43

    accuracy                           0.66       206
   macro avg       0.54      0.55      0.54       206
weighted avg       0.70      0.66      0.68       206

max_abs
[[98 65]
 [17 26]]
              precision    recall  f1-score   support

           0       0.85      0.60      0.71       163
           1       0.29      0.60      0.39        43

    accuracy                           0.60       206
   macro avg       0.57      0.60      0.55       206
weighted avg       0.73      0.60      0.64       206



/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


min_max
[[120  43]
 [ 23  20]]
              precision    recall  f1-score   support

           0       0.84      0.74      0.78       163
           1       0.32      0.47      0.38        43

    accuracy                           0.68       206
   macro avg       0.58      0.60      0.58       206
weighted avg       0.73      0.68      0.70       206



/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/py

robust
[[117  46]
 [ 31  12]]
              precision    recall  f1-score   support

           0       0.79      0.72      0.75       163
           1       0.21      0.28      0.24        43

    accuracy                           0.63       206
   macro avg       0.50      0.50      0.50       206
weighted avg       0.67      0.63      0.64       206



/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/lucas/miniconda3/envs/pluto/lib/py

standard
[[108  55]
 [ 19  24]]
              precision    recall  f1-score   support

           0       0.85      0.66      0.74       163
           1       0.30      0.56      0.39        43

    accuracy                           0.64       206
   macro avg       0.58      0.61      0.57       206
weighted avg       0.74      0.64      0.67       206



/home/lucas/miniconda3/envs/pluto/lib/python3.14/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=50000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


### Resultados de SVC para PCA:

Los datasets pasados por PCA no han sido capaces de arrojar mejores resultados una SVC que con el modelo lineal.